In [1]:
# Import TensorFlow
import tensorflow as tf

# Import NumPy
import numpy as np

# Import Pandas
import pandas as pd

# Import OS
import os

# Import Scikit-learn utilities
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Import plotting libraries
import matplotlib.pyplot as plt

print("All libraries imported successfully!")

print("TensorFlow version:", tf.__version__)

print("GPU devices:", tf.config.list_physical_devices("GPU"))

All libraries imported successfully!
TensorFlow version: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print("\nFolder:", root)

    for file in files[:10]:
        print("  File:", file)


Folder: /kaggle/input

Folder: /kaggle/input/competitions

Folder: /kaggle/input/competitions/aptos2019-blindness-detection
  File: sample_submission.csv
  File: train.csv
  File: test.csv

Folder: /kaggle/input/competitions/aptos2019-blindness-detection/train_images
  File: ef476be214d4.png
  File: 6dcde47060f9.png
  File: ec363f48867b.png
  File: 17f6c7072f61.png
  File: b49b2fac2514.png
  File: af6166d57f13.png
  File: 8d13c46e7d75.png
  File: c3b15bf9b4bc.png
  File: be68322c7223.png
  File: 88e4399d207c.png

Folder: /kaggle/input/competitions/aptos2019-blindness-detection/test_images
  File: 755615db51d3.png
  File: d8f6c660bf2d.png
  File: 619ce2c77917.png
  File: 94a6e4f3c625.png
  File: ee506d853270.png
  File: acd52452b9a2.png
  File: b0b0d564e6c6.png
  File: 228769e96741.png
  File: a87e705d63ee.png
  File: 8c49d9de02d6.png

Folder: /kaggle/input/models

Folder: /kaggle/input/models/hemasakshis

Folder: /kaggle/input/models/hemasakshis/dr-intial-model

Folder: /kaggle/input/

In [14]:
# Dataset directory
DATA_DIR = "/kaggle/input/competitions/aptos2019-blindness-detection"

# CSV file containing training labels
CSV_PATH = os.path.join(DATA_DIR, "train.csv")

# Folder containing retinal images
IMAGE_DIR = os.path.join(DATA_DIR, "train_images")

# Previously trained ResNet50 model
MODEL_PATH = "/kaggle/input/models/hemasakshis/dr-intial-model/tensorflow2/default/1/best_resnet50.keras"

# Check whether all paths exist
print("CSV exists:", os.path.exists(CSV_PATH))
print("Image folder exists:", os.path.exists(IMAGE_DIR))
print("Model exists:", os.path.exists(MODEL_PATH))

CSV exists: True
Image folder exists: True
Model exists: True


In [5]:
# Load the CSV file
df = pd.read_csv(CSV_PATH)

# Display the first five rows
print("First five rows:")
display(df.head())

# Display dataset information
print("\nDataset shape:", df.shape)

# Display column names
print("\nColumn names:", df.columns.tolist())

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

First five rows:


,id_code,diagnosis
0,000c1434d8d7,2
1,001639a390f0,4
2,0024cdab0c1e,1
3,002c21358ce6,0
4,005b95c28852,0



Dataset shape: (3662, 2)

Column names: ['id_code', 'diagnosis']

Missing values:
id_code      0
diagnosis    0
dtype: int64


In [6]:
# Create complete image paths
df["image_path"] = df["id_code"].apply(
    lambda x: os.path.join(IMAGE_DIR, x + ".png")
)

# Extract image paths
image_paths = df["image_path"].values

# Extract diagnosis labels
labels = df["diagnosis"].values

print("Total images:", len(image_paths))
print("Total labels:", len(labels))

print("\nFirst 5 image paths:")
print(image_paths[:5])

print("\nFirst 5 labels:")
print(labels[:5])

# Check whether all image files exist
print("\nAll images exist:", all(os.path.exists(p) for p in image_paths))

Total images: 3662
Total labels: 3662

First 5 image paths:
['/kaggle/input/competitions/aptos2019-blindness-detection/train_images/000c1434d8d7.png'
 '/kaggle/input/competitions/aptos2019-blindness-detection/train_images/001639a390f0.png'
 '/kaggle/input/competitions/aptos2019-blindness-detection/train_images/0024cdab0c1e.png'
 '/kaggle/input/competitions/aptos2019-blindness-detection/train_images/002c21358ce6.png'
 '/kaggle/input/competitions/aptos2019-blindness-detection/train_images/005b95c28852.png']

First 5 labels:
[2 4 1 0 0]

All images exist: True


In [7]:
# First split: 70% training and 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths,
    labels,
    test_size=0.30,
    random_state=42,
    stratify=labels
)

# Second split: Divide temporary data equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training images:", len(X_train))
print("Validation images:", len(X_val))
print("Testing images:", len(X_test))

Training images: 2563
Validation images: 549
Testing images: 550


In [8]:
# Import image preprocessing tools
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import preprocess_input

# Define image size
IMG_SIZE = 224

# Function to load and preprocess an image
def preprocess_image(image_path):

    # Load image and resize it
    img = load_img(
        image_path,
        target_size=(IMG_SIZE, IMG_SIZE)
    )

    # Convert image into NumPy array
    img = img_to_array(img)

    # Apply ResNet50 preprocessing
    img = preprocess_input(img)

    return img

print("Image preprocessing function created successfully!")

Image preprocessing function created successfully!


In [9]:
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 32

# Function to load an image and its label
def load_and_preprocess_image(image_path, label):

    # Read image file
    image = tf.io.read_file(image_path)

    # Decode PNG image into RGB format
    image = tf.image.decode_png(image, channels=3)

    # Resize image to 224 x 224
    image = tf.image.resize(image, [224, 224])

    # Apply ResNet50 preprocessing
    image = tf.keras.applications.resnet50.preprocess_input(image)

    # Convert label to integer format
    label = tf.cast(label, tf.int32)

    return image, label


# Create training dataset
train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)
)

train_dataset = train_dataset.shuffle(
    buffer_size=len(X_train),
    seed=42
)

train_dataset = train_dataset.map(
    load_and_preprocess_image,
    num_parallel_calls=AUTOTUNE
)

train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)


# Create validation dataset
val_dataset = tf.data.Dataset.from_tensor_slices(
    (X_val, y_val)
)

val_dataset = val_dataset.map(
    load_and_preprocess_image,
    num_parallel_calls=AUTOTUNE
)

val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)


# Create testing dataset
test_dataset = tf.data.Dataset.from_tensor_slices(
    (X_test, y_test)
)

test_dataset = test_dataset.map(
    load_and_preprocess_image,
    num_parallel_calls=AUTOTUNE
)

test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)


print("Training, validation and testing datasets created!")

Training, validation and testing datasets created!


In [10]:
for images, labels_batch in train_dataset.take(1):

    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels_batch.shape)
    print("Image data type:", images.dtype)
    print("Label data type:", labels_batch.dtype)
    print("Pixel value range:",
          tf.reduce_min(images).numpy(),
          "to",
          tf.reduce_max(images).numpy())

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
Image data type: <dtype: 'float32'>
Label data type: <dtype: 'int32'>
Pixel value range: -123.68 to 150.94415


In [15]:
# Load the previously trained model
model = tf.keras.models.load_model(MODEL_PATH)

print("Trained model loaded successfully!")

# Display model summary
model.summary()

Trained model loaded successfully!


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │        10,245 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,618,449 (90.10 MB)

 Trainable params: 10,245 (40.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

 Optimizer params: 20,492 (80.05 KB)

In [13]:
import os

MODEL_DIR = "/kaggle/input/models/hemasakshis/dr-intial-model/tensorflow2/default/1"

for root, dirs, files in os.walk(MODEL_DIR):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/models/hemasakshis/dr-intial-model/tensorflow2/default/1/best_resnet50.keras


In [16]:
classes = np.unique(y_train)

# Calculate class weights
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

# Convert weights into a dictionary
class_weights = dict(zip(classes, weights))

print("Class weights:")
print(class_weights)

Class weights:
{np.int64(0): np.float64(0.4058590657165479), np.int64(1): np.float64(1.9791505791505792), np.int64(2): np.float64(0.7333333333333333), np.int64(3): np.float64(3.797037037037037), np.int64(4): np.float64(2.476328502415459)}


In [17]:
# Access the ResNet50 base model
base_model = model.get_layer("resnet50")

# Allow fine-tuning
base_model.trainable = True

# Freeze the first 140 layers
for layer in base_model.layers[:140]:
    layer.trainable = False

# Count trainable and frozen layers
trainable_layers = sum(
    layer.trainable for layer in base_model.layers
)

frozen_layers = sum(
    not layer.trainable for layer in base_model.layers
)

print("Total ResNet50 layers:", len(base_model.layers))
print("Trainable layers:", trainable_layers)
print("Frozen layers:", frozen_layers)

Total ResNet50 layers: 175
Trainable layers: 35
Frozen layers: 140


In [18]:
# Recompile with a smaller learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.00001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model recompiled successfully!")

Model recompiled successfully!


In [19]:
# Save the best fine-tuned model
fine_tune_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/kaggle/working/finetuned_resnet50.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

# Stop training if validation loss does not improve
fine_tune_early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# Start fine-tuning
fine_tune_history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weights,
    callbacks=[
        fine_tune_checkpoint,
        fine_tune_early_stopping
    ]
)

Epoch 1/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5800 - loss: 1.7505
Epoch 1: val_loss improved from None to 0.89378, saving model to /kaggle/working/finetuned_resnet50.keras

Epoch 1: finished saving model to /kaggle/working/finetuned_resnet50.keras
81/81 ━━━━━━━━━━━━━━━━━━━━ 155s 2s/step - accuracy: 0.6391 - loss: 1.4468 - val_accuracy: 0.6776 - val_loss: 0.8938
Epoch 2/10
80/81 ━━━━━━━━━━━━━━━━━━━━ 0s 857ms/step - accuracy: 0.7486 - loss: 0.8764
Epoch 2: val_loss improved from 0.89378 to 0.77579, saving model to /kaggle/working/finetuned_resnet50.keras

Epoch 2: finished saving model to /kaggle/working/finetuned_resnet50.keras
81/81 ━━━━━━━━━━━━━━━━━━━━ 85s 1s/step - accuracy: 0.7639 - loss: 0.8388 - val_accuracy: 0.7140 - val_loss: 0.7758
Epoch 3/10
80/81 ━━━━━━━━━━━━━━━━━━━━ 0s 857ms/step - accuracy: 0.8125 - loss: 0.6245
Epoch 3: val_loss improved from 0.77579 to 0.72599, saving model to /kaggle/working/finetuned_resnet50.keras

Epoch 3: finished saving model to /ka

In [20]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)

# Generate predictions on validation data
y_pred_prob = model.predict(val_dataset)

# Convert probabilities into predicted classes
y_pred = np.argmax(y_pred_prob, axis=1)

# Display classification report
print("CLASSIFICATION REPORT")

print(classification_report(
    y_val,
    y_pred,
    target_names=[
        "No DR",
        "Mild",
        "Moderate",
        "Severe",
        "Proliferative DR"
    ]
))

# Calculate Quadratic Weighted Kappa
qwk = cohen_kappa_score(
    y_val,
    y_pred,
    weights="quadratic"
)

print("Quadratic Weighted Kappa:", qwk)

# Display confusion matrix
print("\nCONFUSION MATRIX")
print(confusion_matrix(y_val, y_pred))

18/18 ━━━━━━━━━━━━━━━━━━━━ 21s 862ms/step
CLASSIFICATION REPORT
                  precision    recall  f1-score   support

           No DR       0.94      0.96      0.95       271
            Mild       0.48      0.56      0.52        55
        Moderate       0.72      0.68      0.70       150
          Severe       0.43      0.55      0.48        29
Proliferative DR       0.59      0.43      0.50        44

        accuracy                           0.78       549
       macro avg       0.64      0.64      0.63       549
    weighted avg       0.78      0.78      0.78       549

Quadratic Weighted Kappa: 0.8350179539013965

CONFUSION MATRIX
[[259   6   6   0   0]
 [  9  31  14   0   1]
 [  7  18 102  15   8]
 [  0   1   8  16   4]
 [  0   8  11   6  19]]


In [21]:
# Save the current fine-tuned model
model.save("/kaggle/working/finetuned_35_layers.keras")

print("Experiment A model saved successfully!")

Experiment A model saved successfully!


In [22]:
base_model = model.get_layer("resnet50")

# Allow fine-tuning
base_model.trainable = True

# Freeze the first 115 layers
for layer in base_model.layers[:115]:
    layer.trainable = False

# Keep Batch Normalization layers frozen
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

print("Total ResNet50 layers:", len(base_model.layers))
print("Trainable layers:", sum(layer.trainable for layer in base_model.layers))

Total ResNet50 layers: 175
Trainable layers: 24


In [23]:
print("Total layers:", len(base_model.layers))

print("Trainable layers:",
      sum(layer.trainable for layer in base_model.layers))

print("Frozen layers:",
      sum(not layer.trainable for layer in base_model.layers))

print("Batch Normalization layers:",
      sum(isinstance(layer, tf.keras.layers.BatchNormalization)
          for layer in base_model.layers))

Total layers: 175
Trainable layers: 24
Frozen layers: 151
Batch Normalization layers: 53


In [24]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Experiment B model compiled successfully!")

Experiment B model compiled successfully!


In [25]:
fine_tune_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/kaggle/working/finetuned_60_layers.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

fine_tune_early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print("Experiment B callbacks created successfully!")

Experiment B callbacks created successfully!


In [27]:
history_B = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weights,
    callbacks=[
        fine_tune_checkpoint,
        fine_tune_early_stopping
    ]
)

Epoch 1/10
80/81 ━━━━━━━━━━━━━━━━━━━━ 0s 855ms/step - accuracy: 0.9412 - loss: 0.1979
Epoch 1: val_loss improved from None to 0.78786, saving model to /kaggle/working/finetuned_60_layers.keras

Epoch 1: finished saving model to /kaggle/working/finetuned_60_layers.keras
81/81 ━━━━━━━━━━━━━━━━━━━━ 89s 1s/step - accuracy: 0.9458 - loss: 0.2007 - val_accuracy: 0.7832 - val_loss: 0.7879
Epoch 2/10
80/81 ━━━━━━━━━━━━━━━━━━━━ 0s 855ms/step - accuracy: 0.9517 - loss: 0.1682
Epoch 2: val_loss did not improve from 0.78786
81/81 ━━━━━━━━━━━━━━━━━━━━ 84s 1s/step - accuracy: 0.9614 - loss: 0.1494 - val_accuracy: 0.7832 - val_loss: 0.8020
Epoch 3/10
80/81 ━━━━━━━━━━━━━━━━━━━━ 0s 857ms/step - accuracy: 0.9578 - loss: 0.1579
Epoch 3: val_loss did not improve from 0.78786
81/81 ━━━━━━━━━━━━━━━━━━━━ 84s 1s/step - accuracy: 0.9653 - loss: 0.1415 - val_accuracy: 0.7832 - val_loss: 0.8152
Epoch 4/10
80/81 ━━━━━━━━━━━━━━━━━━━━ 0s 857ms/step - accuracy: 0.9778 - loss: 0.0941
Epoch 4: val_loss did not improve

In [28]:
model_B = tf.keras.models.load_model(
    "/kaggle/working/finetuned_60_layers.keras"
)

print("Best Experiment B model loaded successfully!")

Best Experiment B model loaded successfully!


In [29]:
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
import numpy as np

y_true = np.concatenate([y.numpy() for x, y in val_dataset])

y_pred_prob_B = model_B.predict(val_dataset)
y_pred_B = np.argmax(y_pred_prob_B, axis=1)

print("EXPERIMENT B - CLASSIFICATION REPORT")

print(classification_report(
    y_true,
    y_pred_B,
    target_names=[
        "No DR",
        "Mild",
        "Moderate",
        "Severe",
        "Proliferative DR"
    ]
))

print("CONFUSION MATRIX")
print(confusion_matrix(y_true, y_pred_B))

print("QWK:",
      cohen_kappa_score(y_true, y_pred_B, weights="quadratic"))

18/18 ━━━━━━━━━━━━━━━━━━━━ 21s 865ms/step
EXPERIMENT B - CLASSIFICATION REPORT
                  precision    recall  f1-score   support

           No DR       0.95      0.96      0.95       271
            Mild       0.56      0.51      0.53        55
        Moderate       0.70      0.75      0.72       150
          Severe       0.39      0.59      0.47        29
Proliferative DR       0.67      0.32      0.43        44

        accuracy                           0.78       549
       macro avg       0.65      0.62      0.62       549
    weighted avg       0.79      0.78      0.78       549

CONFUSION MATRIX
[[259   6   6   0   0]
 [ 10  28  17   0   0]
 [  5  11 112  18   4]
 [  0   0   9  17   3]
 [  0   5  16   9  14]]
QWK: 0.8537179544113636


In [30]:
model_B.save("/kaggle/working/finetuned_60_layers.keras")

print("Experiment B model saved successfully!")

Experiment B model saved successfully!


In [31]:
import os

model_path = "/kaggle/working/finetuned_60_layers.keras"

print("File exists:", os.path.exists(model_path))

if os.path.exists(model_path):
    print("File size:", os.path.getsize(model_path) / (1024 * 1024), "MB")

File exists: True
File size: 204.81947803497314 MB


In [33]:
import os

print("Working directory contents:\n")

for item in os.listdir("/kaggle/working"):
    path = os.path.join("/kaggle/working", item)

    if os.path.isfile(path):
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"FILE: {item} | {size:.2f} MB")
    else:
        print(f"FOLDER: {item}")

Working directory contents:

FILE: finetuned_35_layers.keras | 205.02 MB
FILE: finetuned_resnet50.keras | 205.02 MB
FOLDER: .virtual_documents
FILE: finetuned_60_layers.keras | 204.82 MB
